In [773]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [774]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [775]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [776]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['ECOM', 'ECOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [777]:
realignment_df

,psku realignment master,parent material_code old,asm,channel,new parent_material code,deletion indicator?,psku old,psku new
1,726067_ALL_E-Commerce,726067_SMO CLS MSL 500G MILLET MT ECOM B2C,ALL,ECOM,730975_SMO CLS MSL 550G MILLET MT ECOM,None,726067,730975
8,726065_ALL_E-Commerce,726065_SMO VEG TWST 500G MILLET MT ECOM,ALL,ECOM,731006_SMO VEG TWST 550G MILLET MT ECOM,None,726065,731006
9,726066_ALL_E-Commerce,726066_SMO PEP TOM 500G MILLET MT ECOM,ALL,ECOM,731008_SMO PEP TOM 550G MILLET MT ECOM,None,726066,731008
10,718574_ALL_E-Commerce,718574_SMO MSL&COR 500g PCH,ALL,ECOM,731007_SMO MAS COR 550G MILLET MT ECOM,None,718574,731007
135,718588_ALL_E-Commerce,718588_SETWET HAIRGEL ULTIMATE HOLD 250ml JAR,ALL,ECOM,730277_SW SPORTS EXTREM 250ML MT ECOM NF,None,718588,730277
190,726067_ALL_E-Commerce,None,ALL,ECOM,None,None,726067,730975
197,726065_ALL_E-Commerce,None,ALL,ECOM,None,None,726065,731006
198,726066_ALL_E-Commerce,None,ALL,ECOM,None,None,726066,731008
199,718574_ALL_E-Commerce,None,ALL,ECOM,None,None,718574,731007
276,718588_ALL_E-Commerce,None,ALL,ECOM,None,None,718588,730277


In [778]:
from tqdm import tqdm

def impute_missing_dates(
        df,
        freq='M',
        key=['CHAIN', 'PARENT_MATERIAL_CODE'], 
        date_col='MONTH_DATE',
        max_date='2026-12-31'
    ):

    impute_df = df.copy()
    impute_df[date_col] = pd.to_datetime(impute_df[date_col])
    impute_df['key'] = impute_df[key].astype(str).agg('_'.join, axis=1)
    min_dates_df = impute_df.groupby(
        'key', as_index=False
    )[date_col].min()

    def impute_missing_dates_key(key, min_date, max_date):
        df_imputed = pd.DataFrame(
            pd.date_range(min_date, max_date, freq=freq),
            columns=[date_col]
        )
        df_imputed['key'] = key
        return df_imputed
    
    outputs = Parallel(n_jobs=-1)(
        delayed(impute_missing_dates_key)(row['key'], row[date_col], max_date)
        for idx, row in tqdm(min_dates_df.iterrows())
    )
    df_full = pd.concat(outputs)
    df_full.reset_index(drop=True, inplace=True)

    df_full = df_full.merge(
        impute_df, on=['key', date_col], how='left',
    )

    return df_full

### Primary Actuals + Sec Plan Data

In [779]:
plan_actuals_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Purplle' THEN 'Purplle'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

plan_actuals_df = pd.read_sql(
    plan_actuals_query,
    prod_conn
)

In [780]:
top_chains = ['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Myntra', 'Nykaa', 'Purplle']

In [781]:
plan_actuals_df['CHAIN'].unique()

array(['1MG', 'Amazon ARIPL', 'Amazon RK', 'Big Basket', 'City Mall',
       'Dealshare', 'EMAZING DEALS', 'FATEHPURIA HYGIENE', 'First Cry',
       'Flipkart Grocery', 'Flipkart National', 'Meesho', 'Myntra',
       'Nykaa', 'Purplle'], dtype=object)

In [782]:
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = pd.to_datetime(plan_actuals_df['month_date'])

In [783]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

0

In [784]:
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), column='parent_material_code')

In [785]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

536

In [786]:
plan_actuals_df['month_date'] = plan_actuals_df['month_date'] + MonthEnd(0)

In [787]:
plan_actuals_df = plan_actuals_df[
    (plan_actuals_df['parent_material_code'] != 715096)
]

In [788]:
715096 in plan_actuals_df['parent_material_code'].values

False

In [789]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [790]:
plan_actuals_df.shape

(144500, 8)

In [791]:
plan_actuals_df = impute_missing_dates(
    plan_actuals_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

10874it [00:03, 3232.82it/s]


In [792]:
cols = ['chain', 'parent_material_code', 'material_group_code']

plan_actuals_df[cols] = plan_actuals_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [793]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].fillna(0)

In [794]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [795]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

52

In [796]:
plan_actuals_df.shape

(374859, 9)

In [797]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = plan_actuals_df[plan_actuals_df.duplicated(subset=['key', 'month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
plan_actuals_df = plan_actuals_df.drop(indices_to_drop)

# Verify no duplicates remain
print(plan_actuals_df.duplicated(subset=['key', 'month_date']).sum())

0


In [798]:
plan_actuals_df.shape

(374807, 9)

In [799]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [800]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].clip(lower=0)

In [801]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [802]:
plan_actuals_df.sort_values(['key', 'month_date'], inplace=True)

In [803]:
plan_actuals_df['Primary P3M'] = plan_actuals_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [804]:
others_df = plan_actuals_df[
    ~plan_actuals_df['chain'].isin(top_chains)
]
plan_actuals_df = plan_actuals_df[
    plan_actuals_df['chain'].isin(top_chains)
]

In [805]:
print(others_df['chain'].unique())
print(plan_actuals_df['chain'].unique())

['1MG' 'City Mall' 'Dealshare' 'EMAZING DEALS' 'FATEHPURIA HYGIENE'
 'First Cry' 'Meesho']
['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Myntra' 'Nykaa' 'Purplle']


### Offtakes

In [806]:
offtakes_monthly_query = """
SELECT
    OTM.platform_name AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    OTM.month_date,
    SUM(vol_in_rum) AS vol_in_roum
FROM 
    dwh_ecommplatform_offtake OTM 
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON OTM.material_code = MM.material_code
WHERE
    OTM.platform_name NOT IN ('Blinkit', 'Swiggy', 'Zepto', 'Others', 'Amazon D2C', 'Flipkart D2C') AND
    OTM.month_date > '2022-12-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
""" 

offtakes_monthly_df = pd.read_sql(
    offtakes_monthly_query,
    prod_conn
)

In [807]:
offtakes_monthly_df.columns = offtakes_monthly_df.columns.str.lower()

In [808]:
offtakes_monthly_df['month_date'] = pd.to_datetime(offtakes_monthly_df['month_date'])
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [809]:
offtakes_monthly_df.rename(columns={'vol_in_roum': 'offtake_vol_rum'}, inplace=True)

In [810]:
offtakes_monthly_df

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum
0,Amazon,710542,PCNO(R),2024-09-30,0.450
1,Amazon,710542,PCNO(R),2024-10-31,0.540
2,Amazon,718288,SAFF GOLD,2023-01-31,9.640
3,Amazon,718288,SAFF GOLD,2023-02-28,7.915
4,Amazon,718288,SAFF GOLD,2023-03-31,9.505
...,...,...,...,...,...
56393,Purplle,810673,PA_ESS_HO,2026-02-28,0.126
56394,Purplle,810673,PA_ESS_HO,2026-03-31,0.056
56395,Purplle,810673,PA_ESS_HO,2026-04-30,0.070
56396,Purplle,810674,PA_ESS_HO,2026-02-28,0.224


In [811]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-04-30 00:00:00')

In [812]:
mmonth_df = pd.read_sql("""
    SELECT * 
    FROM 
        TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU
    WHERE
        month_date = '2026-04-30' AND
        run_month='2026-05-31'
""", 
    dev_conn
)

In [813]:
mmonth_df.columns = mmonth_df.columns.str.lower()

In [814]:
mmonth_df['month_date'] = pd.to_datetime(mmonth_df['month_date'])
mmonth_df['parent_material_code'] = mmonth_df['parent_material_code'].astype(int)

In [815]:
mmonth_df.head()

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month
0,2026-04-30,Amazon ARIPL,718288,SAFF GOLD,12.1020,16.805474,0,0,0,0,0,0,0,0,0,0,0,2026-05-31
1,2026-04-30,Amazon ARIPL,718321,SAFF KO,0.0000,0.000000,1,0,0,0,0,0,0,0,0,0,0,2026-05-31
2,2026-04-30,Amazon ARIPL,718322,SAFF KO,15.0850,25.467634,0,0,0,0,0,0,0,0,0,0,0,2026-05-31
3,2026-04-30,Amazon ARIPL,718323,SF_IMV_MK,0.0000,0.000000,1,0,0,0,0,0,0,0,0,0,0,2026-05-31
4,2026-04-30,Amazon ARIPL,718328,SAFF KOCO,9.4761,11.715955,0,0,0,0,0,0,0,0,0,0,0,2026-05-31


In [816]:
mmonth_df = mmonth_df.groupby(
    ['platform_name', 'parent_material_code', 'brand_code', 'month_date'],
    as_index=False
)['vol_in_rum'].sum()

In [817]:
mmonth_df.rename(
    columns = {'platform_name': 'chain', 'brand_code': 'material_group_code', 'vol_in_rum': 'offtake_vol_rum'},
    inplace=True
)

In [818]:
offtakes_monthly_df = pd.concat([
    offtakes_monthly_df,
    mmonth_df
])

In [819]:
offtakes_monthly_df[offtakes_monthly_df['month_date'] == '2026-02-28']['offtake_vol_rum'].sum()

351669.82654000004

In [820]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

964

In [821]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.head()

,brand_code,portfolio
0,ADV-AHO-R,Hair Oils
1,ADV-COL-R,Hair Oils
2,ADV-COL-S,Hair Oils
3,BD_BDOL_M,Male Grooming
4,BD_HRWX_M,Male Grooming


In [822]:
len_before_merge = len(offtakes_monthly_df)
offtakes_monthly_df = offtakes_monthly_df.merge(
    brand_md_df.rename(columns={'brand_code': 'material_group_code'}),
    on=['material_group_code'],
    how='left'
)
assert len_before_merge == len(offtakes_monthly_df)

In [823]:
offtakes_monthly_df[offtakes_monthly_df['portfolio'].isna()]

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio
7887,Amazon,732778,SFOATS-PR,2026-02-28,0.6910,NaN
7888,Amazon,732778,SFOATS-PR,2026-03-31,1.0330,NaN
7889,Amazon,732778,SFOATS-PR,2026-04-30,1.6650,NaN
7890,Amazon,732779,SFOATS-PR,2026-03-31,0.0412,NaN
7891,Amazon,732779,SFOATS-PR,2026-04-30,0.2312,NaN
7892,Amazon,732901,PA_SHMP_R,2026-04-30,118.3200,NaN
7893,Amazon,732904,PA_SHMP_R,2026-04-30,162.0000,NaN
7894,Amazon,732907,PA_SHMP_R,2026-04-30,109.1400,NaN
7895,Amazon,732924,PA_SHMP_R,2026-04-30,226.8000,NaN
12703,Amazon,811287,PA_RSW_SR,2026-04-30,0.3000,NaN


In [824]:
offtakes_monthly_df['chain'] = np.where(
    (offtakes_monthly_df['chain'] == 'Amazon'),
    np.where(
        offtakes_monthly_df['portfolio'].isin(['Saffola Oils', 'Foods']),
        'Amazon ARIPL',
        'Amazon RK'
    ),
    offtakes_monthly_df['chain']
)

In [825]:
offtakes_monthly_df[offtakes_monthly_df['chain'].str.startswith('Amazon')][['chain', 'portfolio']].drop_duplicates()

,chain,portfolio
0,Amazon RK,CNO
2,Amazon ARIPL,Saffola Oils
119,Amazon RK,Hair Oils
213,Amazon ARIPL,Foods
216,Amazon RK,Others
310,Amazon RK,Male Grooming
409,Amazon RK,Prem. Hair Nour.
991,Amazon RK,Skin Care
7887,Amazon RK,NaN


In [826]:
offtakes_monthly_df['chain'].unique()

array(['Amazon RK', 'Amazon ARIPL', 'Big Basket', 'Flipkart Grocery',
       'Flipkart Minutes', 'Flipkart National', 'Meesho', 'Myntra',
       'Nykaa', 'Purplle'], dtype=object)

In [827]:
del offtakes_monthly_df['portfolio']

In [828]:
offtakes_monthly_df = realign_pskus(offtakes_monthly_df.copy(), 'parent_material_code')

In [829]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

1238

In [830]:
offtakes_monthly_df = offtakes_monthly_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'],
    as_index=False
).sum()

In [831]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [832]:
offtakes_monthly_df.head()

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum
0,Amazon ARIPL,718288,SAFF GOLD,2023-01-31,9.640
1,Amazon ARIPL,718288,SAFF GOLD,2023-02-28,7.915
2,Amazon ARIPL,718288,SAFF GOLD,2023-03-31,9.505
3,Amazon ARIPL,718288,SAFF GOLD,2023-04-30,9.290
4,Amazon ARIPL,718288,SAFF GOLD,2023-05-31,8.050


In [833]:
offtakes_monthly_df = impute_missing_dates(
    offtakes_monthly_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

3509it [00:00, 5679.70it/s]


In [834]:
offtakes_monthly_df.sort_values(by=['key', 'month_date'], inplace=True)

cols = ['chain', 'parent_material_code', 'material_group_code']

offtakes_monthly_df[cols] = offtakes_monthly_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [835]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [836]:
offtakes_monthly_df['offtake_vol_rum'] = offtakes_monthly_df['offtake_vol_rum'].fillna(0)

In [837]:
offtakes_monthly_df['offtake_vol_rum'].min()

0.0

In [838]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [839]:
# offtakes_monthly_df['key'] = offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [840]:
(offtakes_monthly_df['key'] == offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [841]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
123626,2026-08-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-08-31
123627,2026-09-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-09-30
123628,2026-10-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-10-31
123629,2026-11-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-11-30


In [842]:
offtakes_monthly_df.duplicated(subset=['key', 'month_date']).sum()

0

In [843]:
715096 in offtakes_monthly_df['parent_material_code'].values

True

In [844]:
offtakes_monthly_df = offtakes_monthly_df[
    offtakes_monthly_df['parent_material_code'] != 715096
]

In [845]:
offtakes_monthly_df['offtake_vol_rum'].min()

0.0

In [846]:
offtakes_monthly_df.to_excel('Offtakes_Chain_PSKU_ECOM.xlsx', index=False)

In [847]:
offtakes_monthly_df.groupby(['month_date'])['offtake_vol_rum'].sum()[30:]

month_date
2025-07-31    278377.263560
2025-08-31    258971.933770
2025-09-30    544883.557660
2025-10-31    523456.543610
2025-11-30    483400.337140
2025-12-31    496069.679560
2026-01-31    450861.594850
2026-02-28    351669.826540
2026-03-31    366449.897800
2026-04-30    652237.271512
2026-05-31         0.000000
2026-06-30         0.000000
2026-07-31         0.000000
2026-08-31         0.000000
2026-09-30         0.000000
2026-10-31         0.000000
2026-11-30         0.000000
2026-12-31         0.000000
Name: offtake_vol_rum, dtype: float64

### SOH

In [848]:
soh_df = pd.read_csv('/data/aman_singh/acuuracy_check/soh_base_jun_run.csv')

In [849]:
soh_df.columns = soh_df.columns.str.lower()

In [850]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,vol_in_lit,vol in rum,indexrate,indexbpm,club sku,roum,roum divide,month,day,month.1
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87246,Amazon ARIPL,B0FS6GCRDG,2.0,721916,SAFF GOLD RC 4L JAR,SAFF GOLD,KL,4000.0,0.008000,137662.938527,...,NaN,NaN,NaN,NaN,4 LTR,NaN,NaN,NaN,NaN,NaN
87247,Amazon ARIPL,B0FS6P6VTW,4.0,721914,SAF GOLD 4L JAR,SAFF GOLD,KL,4000.0,0.016000,137662.938527,...,NaN,NaN,NaN,NaN,4 LTR,NaN,NaN,NaN,NaN,NaN
87248,Amazon ARIPL,B0G1SKX4CM,5408.0,811179,SAFF CDPRS CNO 1L,SAF_CDPRS,KL,1000.0,5.408000,330000.000000,...,NaN,NaN,NaN,NaN,1 LTR,NaN,NaN,NaN,NaN,NaN
87249,Amazon ARIPL,B0GGHLBJL3,854.0,732780,SF HIGH PROTEIN OAT 1KG PC,SFOATS-PR,TO,1000.0,0.854000,430000.000000,...,NaN,NaN,NaN,NaN,1KG,NaN,NaN,NaN,NaN,NaN


In [851]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-06-02
Amazon RK            2026-06-01
BB                   2025-03-22
Big Basket           2026-06-01
Blinkit              2026-06-01
Flipkart Grocery     2026-06-01
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-06-01
Flipkart National    2026-06-01
Meesho               2026-06-01
Myntra               2026-06-01
Nykaa                2026-06-01
Purplle              2025-11-17
Swiggy               2026-06-01
Zepto                2026-06-01
Name: date, dtype: object

In [852]:
import numpy as np
soh_df['date'] = np.where(
    soh_df['date'] > '2026-05-31',
    '2026-05-31',
    soh_df['date']
)

In [853]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-05-31
Amazon RK            2026-05-31
BB                   2025-03-22
Big Basket           2026-05-31
Blinkit              2026-05-31
Flipkart Grocery     2026-05-31
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-05-31
Flipkart National    2026-05-31
Meesho               2026-05-31
Myntra               2026-05-31
Nykaa                2026-05-31
Purplle              2025-11-17
Swiggy               2026-05-31
Zepto                2026-05-31
Name: date, dtype: object

In [854]:
material_master = pd.read_sql("""
SELECT * 
FROM
    mst_material
WHERE
    company_code='MIL' AND
    latest_record_ind=1
""",
    prod_conn
)

In [855]:
material_master.columns = material_master.columns.str.lower()
material_master['material_code'] = material_master['material_code'].astype(int)

In [856]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1'],
      dtype='object')

In [857]:
material_master.columns

Index(['company_code', 'material_code', 'material_desc', 'material_group_code',
       'material_group_desc', 'division_code', 'division_name', 'uom_base',
       'uom_weight', 'gross_weight', 'net_weight', 'parent_material_desc',
       'material_type', 'material_type_desc', 'uom_sales', 'uom_reporting',
       'mg1_code', 'mg1_desc', 'mg2_code', 'mg2_desc', 'mg3_code', 'mg3_desc',
       'mg4_code', 'mg4_desc', 'mg5_code', 'mg5_desc', 'profit_centre_code',
       'vol_per_unit', 'unit_per_case_nbr', 'convert_to_ton', 'convert_to_kl',
       'convert_to_l', 'convert_to_kg', 'convert_to_ml', 'convert_to_gm',
       'csd_id', 'ean_id', 'standard_cost_amt', 'source_system_id',
       'last_bi_updt_date', 'last_src_updt_date', 'material_sk',
       'effective_start_date', 'effective_end_date', 'latest_record_ind',
       'version_nbr', 'parent_material_code', 'parent_material1_code',
       'parent_material1_desc', 'shelf_life_days',
       'manual_material_group_desc', 'manual_mg1_desc',

In [858]:
for sku in soh_df['material_code'].unique():
    try:
        int(sku)
    except:
        print(sku)

Combo


In [859]:
soh_df = soh_df[soh_df['material_code'] != 'Combo']

In [860]:
soh_df['material_code'] = soh_df['material_code'].astype(int)

In [861]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1'],
      dtype='object')

In [862]:
soh_df.drop(['material_group_code', 'vol_per_unit', 'uom_reporting'], axis=1, inplace=True)

In [863]:
len_before_merge = len(soh_df)
soh_df = soh_df.merge(
    material_master[['material_code', 'parent_material_code', 'material_group_code', 'vol_per_unit', 'uom_reporting']],
    on=['material_code'],
    how='left'
)
assert len_before_merge == len(soh_df)
del len_before_merge

In [864]:
# soh_df[soh_df['parent_material_code'].isna()]

In [865]:
soh_df = soh_df[soh_df['parent_material_code'].notna()]

In [866]:
soh_df['parent_material_code'] = soh_df['parent_material_code'].astype(int)

In [867]:
soh_df['chain'] = soh_df['chain'].replace({
    'MYNTRA': 'Myntra',
    'BB': 'Big Basket',
    'Flipkart Mintues': 'Flipkart National',
    'Flipkart Minutes': 'Flipkart National'
})

In [868]:
sorted(soh_df['chain'].unique())

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Blinkit',
 'Flipkart Grocery',
 'Flipkart National',
 'Meesho',
 'Myntra',
 'Nykaa',
 'Purplle',
 'Swiggy',
 'Zepto']

In [869]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,club sku,roum,roum divide,month,day,month.1,parent_material_code,material_group_code,vol_per_unit,uom_reporting
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,718322,SAFF KO,5000.0000,KL
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718492,SFOATS-FL,37.9997,TO
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718494,SFOATS-FL,37.9997,TO
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,807029,SAFF SALT,1000.0000,TO
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,718328,SAFF KOCO,1000.0000,KL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,Amazon ARIPL,B0FS6GCRDG,2.0,721916,SAFF GOLD RC 4L JAR,SAFF GOLD,KL,4000.0,0.008000,137662.938527,...,4 LTR,NaN,NaN,NaN,NaN,NaN,722166,SAFF GOLD,4000.0000,KL
87242,Amazon ARIPL,B0FS6P6VTW,4.0,721914,SAF GOLD 4L JAR,SAFF GOLD,KL,4000.0,0.016000,137662.938527,...,4 LTR,NaN,NaN,NaN,NaN,NaN,722167,SAFF GOLD,4000.0000,KL
87243,Amazon ARIPL,B0G1SKX4CM,5408.0,811179,SAFF CDPRS CNO 1L,SAF_CDPRS,KL,1000.0,5.408000,330000.000000,...,1 LTR,NaN,NaN,NaN,NaN,NaN,811181,SAF_CDPRS,1000.0000,KL
87244,Amazon ARIPL,B0GGHLBJL3,854.0,732780,SF HIGH PROTEIN OAT 1KG PC,SFOATS-PR,TO,1000.0,0.854000,430000.000000,...,1KG,NaN,NaN,NaN,NaN,NaN,732778,SFOATS-PR,1000.0000,TO


In [870]:
soh_df['current_soh'] = soh_df['soh'] * soh_df['vol_per_unit'] / soh_df['uom_reporting'].map(
    lambda x: (10 ** 6) if x in ('KL', 'TO') else (10 ** 3)
)

In [871]:
soh_df.dtypes

chain                    object
fsn                      object
soh                     float64
material_code             int64
desc                     object
brand                    object
uom                      object
vol per unit            float64
vol                     float64
fy index                float64
bpm in lacs             float64
category                 object
ecom brands              object
file                     object
vol in kl               float64
psku                     object
pdes                     object
date                     object
asin                     object
asin.1                   object
platform_name.1          object
product_title            object
ean                     float64
units                   float64
vol_in_lit              float64
vol in rum              float64
indexrate               float64
indexbpm                float64
club sku                 object
roum                    float64
roum divide             float64
month   

In [872]:
soh_df.rename(
    columns={'date': 'inv date'},
    inplace=True
)

In [873]:
soh_df = realign_pskus(soh_df.copy(), 'parent_material_code')

In [874]:
soh_df = soh_df.groupby(
    ['chain', 'parent_material_code', 'inv date'], as_index=False
)['current_soh'].sum().rename(columns={
    'inv date': 'as_on_date'
})

In [875]:
soh_df['key'] = soh_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [876]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])

In [877]:
soh_df['run_month'] = soh_df['as_on_date'] + MonthEnd(0)

In [878]:
soh_df.dtypes

chain                           object
parent_material_code             int64
as_on_date              datetime64[ns]
current_soh                    float64
key                             object
run_month               datetime64[ns]
dtype: object

In [879]:
soh_df

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
...,...,...,...,...,...,...
40735,Zepto,811181,2026-04-01,0.711,Zepto_811181,2026-04-30
40736,Zepto,811181,2026-05-01,0.481,Zepto_811181,2026-05-31
40737,Zepto,811181,2026-05-31,0.216,Zepto_811181,2026-05-31
40738,Zepto,811287,2026-05-01,9.500,Zepto_811287,2026-05-31


In [880]:
date_wise_soh_vol_sum = soh_df.groupby(['as_on_date'], as_index=False)['current_soh'].sum()

In [881]:
date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]

,as_on_date,current_soh


In [882]:
soh_df[
    soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]['current_soh'].sum()

0.0

In [883]:
soh_df = soh_df[
    ~soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]

In [884]:
715096 in soh_df['parent_material_code'].values

False

In [885]:
soh_df[['chain', 'as_on_date']].drop_duplicates().sort_values(by=['chain', 'as_on_date'])

,chain,as_on_date
0,Amazon ARIPL,2024-12-14
1,Amazon ARIPL,2025-01-25
2,Amazon ARIPL,2025-02-22
3,Amazon ARIPL,2025-03-29
4,Amazon ARIPL,2025-04-19
...,...,...
37043,Zepto,2026-01-31
37044,Zepto,2026-02-28
37045,Zepto,2026-04-01
37026,Zepto,2026-05-01


In [886]:
soh_df[soh_df['chain'] == 'Amazon ARIPL'].head(60)

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
5,Amazon ARIPL,718288,2025-05-31,12.048,Amazon ARIPL_718288,2025-05-31
6,Amazon ARIPL,718288,2025-06-19,9.138,Amazon ARIPL_718288,2025-06-30
7,Amazon ARIPL,718288,2025-07-31,6.840,Amazon ARIPL_718288,2025-07-31
8,Amazon ARIPL,718288,2025-08-02,17.118,Amazon ARIPL_718288,2025-08-31
9,Amazon ARIPL,718288,2025-10-05,18.864,Amazon ARIPL_718288,2025-10-31


### Forecasts

In [887]:
forecasts = pd.read_excel(
    r"/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_june_live.xlsx",
    sheet_name='base'
)
forecasts = forecasts[forecasts['key'].notna()]
forecasts.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final Heuristic vol,pred_SARIMA,pred_value_SARIMA,shrink_ratio_sarima,recency_heuristic_sarima,seasonal_heuristic_sarima,non_seasonal_heuristic_sarima,final_heuristic_sarima,final_heuristic_sarima_value,final_heuristic_sarima_value_2
0,Amazon ARIPL_718288,2026-06-30,27.226,24.622,28.546677,25.025422,0.378075,0.341914,0.396414,0.347516,...,0,24.129603,0.335076,1.062226,30.105397,25.680000,23.216869,23.216869,0.322402,0.392938
1,Amazon ARIPL_718288,2026-07-31,27.226,24.622,27.384622,25.037278,0.378075,0.341914,0.380277,0.347681,...,0,23.610279,0.327865,1.073844,30.105397,30.240000,23.204176,23.204176,0.322225,0.384691
2,Amazon ARIPL_718288,2026-08-31,27.226,24.622,27.606416,30.397286,0.378075,0.341914,0.383357,0.422113,...,0,27.762757,0.385528,0.990286,30.105397,27.762757,27.241670,27.241670,0.378292,0.385306
3,Amazon ARIPL_718288,2026-09-30,27.226,24.622,29.646981,25.111586,0.378075,0.341914,0.411694,0.348713,...,0,24.646361,0.342252,1.051031,30.105397,28.732800,23.325277,23.325277,0.323907,0.399488
4,Amazon ARIPL_718288,2026-10-31,27.226,24.622,32.709076,30.133809,0.378075,0.341914,0.454215,0.418454,...,0,27.080109,0.376049,1.002690,30.105397,33.830400,24.451474,24.451474,0.339546,0.418719


In [888]:
forecasts['parent_material_code'] = forecasts['parent_material_code'].astype(int)

In [889]:
forecasts.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf',
       ...
       'final Heuristic vol', 'pred_SARIMA', 'pred_value_SARIMA',
       'shrink_ratio_sarima', 'recency_heuristic_sarima',
       'seasonal_heuristic_sarima', 'non_seasonal_heuristic_sarima',
       'final_heuristic_sarima', 'final_heuristic_sarima_value',
       'final_heuristic_sarima_value_2'],
      dtype='object', length=160)

In [890]:
forecasts['Final Heuristic Prophet Vol'] = forecasts['Final Heuristic value'] * (10 ** 7) / forecasts['qtr_ind_rate_x']
forecasts.rename(columns = {'run_month_x': 'run_month'}, inplace=True)

In [891]:
forecasts = forecasts[['key', 'month_date', 'platform_name', 'parent_material_code', 'brand_code', 
                       'run_month', 'M month', 'Final Heuristic Prophet Vol', 'skipped']]

In [892]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [893]:
forecasts.duplicated(['key', 'run_month', 'month_date']).sum()

0

In [894]:
forecasts[forecasts.duplicated(['key', 'run_month', 'month_date'])]

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped


In [895]:
forecasts['platform_name'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [896]:
# forecasts.rename(columns={'Final Heuristic Prophet 2 Vol': 'Final Heuristic Prophet Vol'}, inplace=True)

In [897]:
top_chains

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Flipkart Grocery',
 'Flipkart National',
 'Myntra',
 'Nykaa',
 'Purplle']

### Collate Everything

In [898]:
print(f"SOH", soh_df['chain'].unique())
print("Offtakes:", offtakes_monthly_df['chain'].unique())
print("Plan Actuals:", plan_actuals_df['chain'].unique())

SOH ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Blinkit' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa' 'Purplle' 'Swiggy' 'Zepto']
Offtakes: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart Minutes' 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa'
 'Purplle']
Plan Actuals: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Myntra' 'Nykaa' 'Purplle']


In [899]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
123626,2026-08-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-08-31
123627,2026-09-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-09-30
123628,2026-10-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-10-31
123629,2026-11-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-11-30


In [900]:
final_df = plan_actuals_df[
    ['key', 'chain', 'parent_material_code', 'material_group_code']
].drop_duplicates()

In [901]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = final_df[final_df.duplicated(subset=['key'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
final_df = final_df.drop(indices_to_drop)

# Verify no duplicates remain
print(final_df.duplicated(subset=['key']).sum())

0


In [902]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [903]:
tmp_df = pd.DataFrame()

for rm in ['2026-06-30']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_df.copy()
        tmp_df2['run_month'] = pd.to_datetime(rm)
        tmp_df2['month_date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_df = tmp_df.copy()    
del tmp_df

### Merge Plan

In [904]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [905]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    plan_actuals_df[['key', 'month_date', 'pri_actuals_vol_rum', 'sec_apo_plan_vol_rum', 'Primary P3M']],
    on=['key', 'month_date'],
    how='left'
)
assert len(final_df) == len_before_merge
del len_before_merge

In [906]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0
1,Amazon ARIPL_715099,Amazon ARIPL,715099,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0
2,Amazon ARIPL_715100,Amazon ARIPL,715100,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0
3,Amazon ARIPL_715106,Amazon ARIPL,715106,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0
4,Amazon ARIPL_715107,Amazon ARIPL,715107,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
65095,Purplle_811268,Purplle,811268,PA_ESS_HO,2026-06-30,2027-02-28,NaN,NaN,NaN
65096,Purplle_811269,Purplle,811269,PA_ESS_HO,2026-06-30,2027-02-28,NaN,NaN,NaN
65097,Purplle_811279,Purplle,811279,SAF_CDPRS,2026-06-30,2027-02-28,NaN,NaN,NaN
65098,Purplle_811287,Purplle,811287,PA_RSW_SR,2026-06-30,2027-02-28,NaN,NaN,NaN


In [907]:
plan_actuals_df[plan_actuals_df['month_date']<'2026-06-30'].to_csv('/data/aman_singh/acuuracy_check/plan_actuals_df_ecom.csv', index=False)

In [908]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [909]:
for col in ['Primary P3M']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

### Merge offtakes

In [910]:
### Monthly Actual Offtakes
len_before_merge = len(final_df)
final_df = final_df.merge(
    offtakes_monthly_df[['month_date', 'key', 'offtake_vol_rum']].rename(columns={
        'offtake_vol_rum': 'Offtake Chain PSKU'
    }),
    on=['month_date', 'key'],
    how='left'
)   
assert len_before_merge == len(final_df)
del len_before_merge

In [911]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
123626,2026-08-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-08-31
123627,2026-09-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-09-30
123628,2026-10-31,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-10-31
123629,2026-11-30,Purplle_810805,Purplle,810805,PABABY_GM,0.000,2026-11-30


In [912]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [913]:
final_df['M month'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(x['month_date'], np.nan),
    axis=1
)

In [914]:
final_df[['run_month', 'month_date', 'M month']].drop_duplicates()

,run_month,month_date,M month
0,2026-06-30,2026-05-31,NaN
1,2026-06-30,2026-06-30,M
2,2026-06-30,2026-07-31,M+1
3,2026-06-30,2026-08-31,M+2
4,2026-06-30,2026-09-30,M+3
5,2026-06-30,2026-10-31,M+4
6,2026-06-30,2026-11-30,M+5
7,2026-06-30,2026-12-31,M+6
8,2026-06-30,2027-01-31,M+7
9,2026-06-30,2027-02-28,M+8


### Merge Forecasts

In [915]:
forecasts.head()

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,Amazon ARIPL_718288,2026-06-30,Amazon ARIPL,718288,SAFF GOLD,2026-06-30,M,28.296346,0
1,Amazon ARIPL_718288,2026-07-31,Amazon ARIPL,718288,SAFF GOLD,2026-06-30,M+1,27.702491,0
2,Amazon ARIPL_718288,2026-08-31,Amazon ARIPL,718288,SAFF GOLD,2026-06-30,M+2,28.767977,0
3,Amazon ARIPL_718288,2026-09-30,Amazon ARIPL,718288,SAFF GOLD,2026-06-30,M+3,29.646981,0
4,Amazon ARIPL_718288,2026-10-31,Amazon ARIPL,718288,SAFF GOLD,2026-06-30,M+4,30.152907,0


In [916]:
forecasts.duplicated(subset=['key', 'month_date', 'run_month']).sum()

0

In [917]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    forecasts[['key', 'month_date', 'run_month', 'Final Heuristic Prophet Vol']],
    on=['key', 'month_date', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [918]:
final_df.rename(columns={'Final Heuristic Prophet Vol': 'Offtake Chain PSKU Forecast Vol'}, inplace=True)

### Norms

In [919]:
soh_df['as_on_date'].min()

Timestamp('2024-12-03 00:00:00')

In [920]:
soh_df[~soh_df['chain'].isin(top_chains)]['chain'].unique()

array(['Blinkit', 'Meesho', 'Swiggy', 'Zepto'], dtype=object)

In [921]:
avg_inventory = soh_df[
    (soh_df['chain'].isin(top_chains))
].groupby(
    ['key'], as_index=False
)['current_soh'].mean()

In [922]:
soh_df['as_on_date'].min(), soh_df['as_on_date'].max()

(Timestamp('2024-12-03 00:00:00'), Timestamp('2026-05-31 00:00:00'))

In [923]:
avg_daily_offtakes = offtakes_monthly_df[
    (offtakes_monthly_df['month_date'] > '2024-11-30') &
    (offtakes_monthly_df['month_date'] < '2026-06-01')
].copy()
avg_daily_offtakes['days'] = avg_daily_offtakes['month_date'].dt.day
avg_daily_offtakes = avg_daily_offtakes.groupby(
    ['key'], as_index=False
)[['offtake_vol_rum', 'days']].sum()

In [924]:
avg_daily_offtakes

,key,offtake_vol_rum,days
0,Amazon ARIPL_718288,320.4900,547
1,Amazon ARIPL_718321,0.0000,547
2,Amazon ARIPL_718322,132.3650,547
3,Amazon ARIPL_718323,0.0000,547
4,Amazon ARIPL_718328,63.8392,547
...,...,...,...
3503,Purplle_809042,1.8000,335
3504,Purplle_809250,0.0198,547
3505,Purplle_810673,0.3080,120
3506,Purplle_810674,0.3640,120


In [925]:
avg_daily_offtakes['avg_offtakes_vol_rum'] = avg_daily_offtakes['offtake_vol_rum'] / avg_daily_offtakes['days']

In [926]:
norm_days = avg_inventory.merge(
    avg_daily_offtakes[['key', 'avg_offtakes_vol_rum']],
    on=['key'],
    how='left'
) 
assert len(norm_days) == len(avg_inventory)

In [927]:
norm_days

,key,current_soh,avg_offtakes_vol_rum
0,Amazon ARIPL_718288,20.907333,0.585905
1,Amazon ARIPL_718312,0.055778,NaN
2,Amazon ARIPL_718322,11.323889,0.241984
3,Amazon ARIPL_718328,3.836833,0.116708
4,Amazon ARIPL_718330,3.418333,0.110119
...,...,...,...
2322,Purplle_807032,14.200000,0.200731
2323,Purplle_807033,6.300000,0.032221
2324,Purplle_807036,0.900000,0.003656
2325,Purplle_809042,16.200000,0.005373


In [928]:
norm_days['norm_days'] = norm_days['current_soh'] / norm_days['avg_offtakes_vol_rum']

In [929]:
# norm_days['norm_days'] = np.minimum(np.maximum(norm_days['norm_days'] - 30, 0), 30)

norm_days['norm_days'] = np.maximum(np.minimum(norm_days['norm_days'], 30), 5)

In [930]:
norm_days['norm_days'] = norm_days['norm_days'].fillna(5)

In [931]:
norms = final_df.copy()

In [932]:
norms = norms[['key', 'run_month', 'month_date', 'Offtake Chain PSKU Forecast Vol']]

In [933]:
norms['total_days_in_month'] = norms['month_date'].dt.day

In [934]:
norms.isnull().sum()

key                                    0
run_month                              0
month_date                             0
Offtake Chain PSKU Forecast Vol    43698
total_days_in_month                    0
dtype: int64

In [935]:
norms

,key,run_month,month_date,Offtake Chain PSKU Forecast Vol,total_days_in_month
0,Amazon ARIPL_715098,2026-06-30,2026-05-31,NaN,31
1,Amazon ARIPL_715098,2026-06-30,2026-06-30,NaN,30
2,Amazon ARIPL_715098,2026-06-30,2026-07-31,NaN,31
3,Amazon ARIPL_715098,2026-06-30,2026-08-31,NaN,31
4,Amazon ARIPL_715098,2026-06-30,2026-09-30,NaN,30
...,...,...,...,...,...
65095,Purplle_811416,2026-06-30,2026-10-31,NaN,31
65096,Purplle_811416,2026-06-30,2026-11-30,NaN,30
65097,Purplle_811416,2026-06-30,2026-12-31,NaN,31
65098,Purplle_811416,2026-06-30,2027-01-31,NaN,31


In [936]:
len_before_merge = len(norms)
norms = norms.merge(
    norm_days[['key', 'norm_days']],
    on=['key'],
    how='left'
)
assert len_before_merge == len(norms)
del len_before_merge

In [937]:
norms['safety_stock'] = norms['Offtake Chain PSKU Forecast Vol'] *  norms['norm_days'] / norms['total_days_in_month']

In [938]:
norm_days[norm_days.duplicated(subset=['key'])]

,key,current_soh,avg_offtakes_vol_rum,norm_days


In [939]:
final_df[final_df['key'] == 'Amazon ARIPL_715098']

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-06-30,0.0,0.0,0.0,NaN,M,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-07-31,0.0,0.0,0.0,NaN,M+1,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-08-31,0.0,0.0,0.0,NaN,M+2,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-09-30,0.0,0.0,0.0,NaN,M+3,NaN
5,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-10-31,0.0,0.0,0.0,NaN,M+4,NaN
6,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-11-30,0.0,0.0,0.0,NaN,M+5,NaN
7,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-12-31,0.0,0.0,0.0,NaN,M+6,NaN
8,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2027-01-31,NaN,NaN,0.0,NaN,M+7,NaN
9,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2027-02-28,NaN,NaN,0.0,NaN,M+8,NaN


In [940]:
final_df[final_df.duplicated(subset=['key', 'run_month', 'month_date'])]#['material_group_code'].unique()

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol


In [941]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month', 'norm_days'], axis=1).rename(columns={
        'safety_stock': 'norms_soh'
    }),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [942]:
norms['month_date'] = norms['month_date'] - MonthEnd(1)

In [943]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month'], axis=1),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [944]:
final_df.sort_values(
    by=['run_month', 'material_group_code', 'key', 'month_date'], inplace=True
)

In [945]:
final_df['safety_stock'] = final_df['safety_stock'].fillna(0)

In [946]:
chain_wise_max_soh_dates = soh_df[soh_df['chain'].isin(top_chains)].groupby(
    ['run_month', 'chain'], as_index=False
)['as_on_date'].max()

chain_wise_max_soh_dates

,run_month,chain,as_on_date
0,2024-12-31,Amazon ARIPL,2024-12-14
1,2024-12-31,Amazon RK,2024-12-19
2,2024-12-31,Big Basket,2024-12-21
3,2024-12-31,Flipkart Grocery,2024-12-16
4,2024-12-31,Flipkart National,2024-12-20
...,...,...,...
110,2026-05-31,Big Basket,2026-05-31
111,2026-05-31,Flipkart Grocery,2026-05-31
112,2026-05-31,Flipkart National,2026-05-31
113,2026-05-31,Myntra,2026-05-31


In [947]:
chain_wise_max_soh_dates = (
    chain_wise_max_soh_dates
    .groupby('chain')
    .apply(lambda x: dict(zip(x['run_month'], x['as_on_date'])))
    .to_dict()
)

In [948]:
last_date_soh = pd.DataFrame()

for c in chain_wise_max_soh_dates.keys():
    for rm in chain_wise_max_soh_dates[c].keys():
        last_date_soh = pd.concat([
            last_date_soh,
            soh_df[
                (soh_df['chain'] == c) &
                (soh_df['as_on_date'] == chain_wise_max_soh_dates[c][rm])
            ]
        ])

In [949]:
last_date_soh['as_on_date'] = last_date_soh['as_on_date'] + MonthEnd(0)

In [950]:
last_date_soh.groupby(['as_on_date'])['current_soh'].sum()

as_on_date
2024-12-31    247940.743731
2025-01-31    315361.659226
2025-02-28    197036.775253
2025-03-31    246659.146418
2025-04-30    367739.756700
2025-05-31    319795.321935
2025-06-30    411340.346692
2025-07-31    117403.153709
2025-08-31    692209.042221
2025-09-30    746045.706095
2025-10-31    362500.123601
2025-11-30    251933.621061
2025-12-31    386497.252399
2026-01-31    438232.389269
2026-02-28     11517.162356
2026-03-31    282498.423831
2026-04-30    238057.427000
2026-05-31    356474.265474
Name: current_soh, dtype: float64

In [951]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    last_date_soh[['key', 'as_on_date', 'current_soh']].rename(
        columns={
            'as_on_date': 'month_date',
            'current_soh': 'Actual Closing SOH'
        }
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)

del len_before_merge

In [952]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [953]:
final_df['Actual Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(1)

final_df['Actual Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(2)

In [954]:
final_df['safety_stock'].sum()

3150138.5720178247

In [955]:
final_df['safety_stock'].min()

0.0

In [956]:
# final_df['Assumed Closing SOH'] = np.where(
#     final_df['M month'] == 'M',  
#     final_df['Actual Closing SOH_Lag_1'].fillna(0) + final_df['sec_apo_plan_vol_rum'].fillna(0) \
#     - final_df['Offtake Chain PSKU Forecast Vol'].fillna(0),
#     final_df['safety_stock']
# )

final_df['Assumed Closing SOH'] = final_df['safety_stock']

In [957]:
final_df['Assumed Closing SOH'] = final_df['Assumed Closing SOH'].clip(lower=0.0)

In [958]:
final_df['Assumed Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(1)

final_df['Assumed Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(2)

In [959]:
final_df = final_df[
    ~final_df['material_group_code'].isin(['NC FREE', 'HC FREE'])
]

In [960]:
final_df.isna().sum()

key                                    0
chain                                  0
parent_material_code                   0
material_group_code                    0
run_month                              0
month_date                             0
pri_actuals_vol_rum                13049
sec_apo_plan_vol_rum               13049
Primary P3M                           29
Offtake Chain PSKU                 43276
M month                             6510
Offtake Chain PSKU Forecast Vol    43698
norms_soh                          49187
norm_days                          45300
safety_stock                           0
Actual Closing SOH                 63721
Actual Closing SOH_Lag_1           63721
Actual Closing SOH Lag 2           63721
Assumed Closing SOH                    0
Assumed Closing SOH_Lag_1           6510
Assumed Closing SOH Lag 2          13020
dtype: int64

In [961]:
final_df.shape

(65100, 21)

### Add P3M, LY

In [962]:
actuals_df = plan_actuals_df.copy()
actuals_df['month_date'] = actuals_df['month_date'] + MonthEnd(0)

In [963]:
actuals_df = actuals_df.groupby(
    ['key', 'month_date'], as_index=False
)[['pri_actuals_vol_rum', 'sec_actuals_vol_rum']].sum()

In [964]:
actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [965]:
actuals_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [966]:
actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [967]:
actuals_df['Primary P3M redundant'] = actuals_df.groupby(
['key'], as_index = False, group_keys = False)['pri_actuals_vol_rum'].shift(1)\
                            .rolling(window=3, min_periods=1).mean()

In [968]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'Primary Actuals Vol',
        'sec_actuals_vol_rum': 'Sec Actuals Vol'
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [969]:
ly_actuals_df = actuals_df.copy()

In [970]:
ly_actuals_df['month_date'] = ly_actuals_df['month_date'] + MonthEnd(12)

In [971]:
ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M'
    })

,key,month_date,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M
0,Amazon ARIPL_715098,2024-07-31,0.0,0.0,NaN
1,Amazon ARIPL_715098,2024-08-31,0.0,0.0,0.0
2,Amazon ARIPL_715098,2024-09-30,0.0,0.0,0.0
3,Amazon ARIPL_715098,2024-10-31,0.0,0.0,0.0
4,Amazon ARIPL_715098,2024-11-30,0.0,0.0,0.0
...,...,...,...,...,...
233742,Purplle_811416,2027-08-31,0.0,0.0,0.0
233743,Purplle_811416,2027-09-30,0.0,0.0,0.0
233744,Purplle_811416,2027-10-31,0.0,0.0,0.0
233745,Purplle_811416,2027-11-30,0.0,0.0,0.0


In [972]:
ly_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [973]:
ly_actuals_df['pri_actuals_vol_rum_lag_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

ly_actuals_df['pri_actuals_vol_rum_lag_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

ly_actuals_df['pri_actuals_vol_rum_lag_3'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)


ly_actuals_df['pri_actuals_vol_rum_lead_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-1)

ly_actuals_df['pri_actuals_vol_rum_lead_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-2)

In [974]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M', 
        'pri_actuals_vol_rum_lag_1': 'LY Primary Actuals Lag 1 Vol',
        'pri_actuals_vol_rum_lag_2': 'LY Primary Actuals Lag 2 Vol',
        'pri_actuals_vol_rum_lag_3': 'LY Primary Actuals Lag 3 Vol',
        'pri_actuals_vol_rum_lead_1': 'LY Primary Actuals Lead 1 Vol',
        'pri_actuals_vol_rum_lead_2': 'LY Primary Actuals Lead 2 Vol',
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [975]:
prev_offtakes_monthly_df = offtakes_monthly_df.copy()

In [976]:
final_offtakes_historical_df = prev_offtakes_monthly_df.copy()

In [977]:
final_offtakes_historical_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [978]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df[['chain', 'parent_material_code']].astype(str).agg(
    '_'.join, axis=1
)

In [979]:

final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [980]:
final_offtakes_historical_df['P3M'] = final_offtakes_historical_df.groupby(
    ['key'], as_index = False, group_keys = False)['offtake_vol_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

In [981]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M']].rename(
        columns={'P3M': 'Offtake P3M', 'offtake_vol_rum': 'Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [982]:
ly_final_offtakes_historical_df = final_offtakes_historical_df.copy()

In [983]:

ly_final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [984]:
ly_final_offtakes_historical_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [985]:
ly_final_offtakes_historical_df['month_date'] = ly_final_offtakes_historical_df['month_date'] + MonthEnd(12)
ly_final_offtakes_historical_df.head()

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month,P3M
0,2024-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31,NaN
1,2024-02-29,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28,NaN
2,2024-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31,NaN
3,2024-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30,9.020000
4,2024-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31,8.903333


In [986]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lag 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 3 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(3)


In [987]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lead 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lead 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-2)

In [988]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M', 
                                     'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol', 
                                     'LY Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lead 1 Vol',
                                     'LY Offtake Actuals Lead 2 Vol']].rename(
        columns={'P3M': 'LY Offtake P3M', 'offtake_vol_rum': 'LY Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [989]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [990]:
for col in ['Primary P3M', 'LY Primary P3M', 'Offtake P3M', 'LY Offtake P3M', 'Primary P3M redundant']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [991]:
final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)

In [992]:
lags_df = plan_actuals_df.copy()

In [993]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
25122,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
25123,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
25124,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
25125,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
25126,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
374854,2026-08-31,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374855,2026-09-30,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374856,2026-10-31,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374857,2026-11-30,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0


In [994]:
lags_df.sort_values(by=['key', 'month_date'], inplace=True)

In [995]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
25122,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
25123,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
25124,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
25125,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
25126,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
374854,2026-08-31,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374855,2026-09-30,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374856,2026-10-31,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
374857,2026-11-30,Purplle_811416,Purplle,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0


In [996]:
lags_df['Primary_Lag_2'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

lags_df['Primary_Lag_3'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

In [997]:
lags_df['month_date'] = lags_df['month_date'] + MonthEnd(1)

In [998]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_df[['key', 'month_date', 'pri_actuals_vol_rum', 'Primary_Lag_2', 'Primary_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'pri_actuals_vol_rum': 'Primary_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [999]:
lags_final_offtakes_historical_df = final_offtakes_historical_df.copy()

lags_final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)
lags_final_offtakes_historical_df['OT_Lag_2'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

lags_final_offtakes_historical_df['OT_Lag_3'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

In [1000]:
lags_final_offtakes_historical_df['month_date'] = lags_final_offtakes_historical_df['month_date'] + MonthEnd(1)

In [1001]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'offtake_vol_rum': 'OT_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [1002]:
# final_df.to_csv('ECOM_OTP_v0_check.csv', index=False)

In [1003]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [1004]:
def calculate_primary_iteratively(key_df):
    key_df = key_df.sort_values('month_date').copy()
    key_df = key_df[key_df['month_date'] >= key_df['run_month']]

    # Fill NaNs
    fill_cols = [
        'Offtake Chain PSKU Forecast Vol',
        'safety_stock',
        'Actual Closing SOH_Lag_1'
    ]
    key_df[fill_cols] = key_df[fill_cols].fillna(0)

    _key = key_df['key'].iloc[0]
    _run_month = key_df['run_month'].iloc[0]

    outputs = []
    prev_soh = None

    for _, row in key_df.iterrows():
        forecast = row['Offtake Chain PSKU Forecast Vol']
        safety_stock = row['safety_stock']

        if row['M month'] == 'M':
            opening_soh = row['Actual Closing SOH_Lag_1']
        else:
            opening_soh = prev_soh

        primary_vol = max(
            forecast + safety_stock - opening_soh,
            0
        )

        assumed_closing_soh = max(
            opening_soh + primary_vol - forecast,
            0
        )

        outputs.append({
            'run_month': _run_month,
            'key': _key,
            'month_date': row['month_date'],
            'Calculated Primary Vol': primary_vol,
            'Final Assumed Closing SOH Vol': assumed_closing_soh
        })

        prev_soh = assumed_closing_soh

    return outputs

In [1005]:
# # final_df['Calculated Primary Vol'] = np.where(
# #     final_df['M month'] == 'M',
# #     final_df['sec_apo_plan_vol_rum'],
# #     final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
# #         final_df['safety_stock'].fillna(0) - \
# #         final_df['Assumed Closing SOH_Lag_1'].fillna(0)
# # )


# final_df['Calculated Primary Vol'] = final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
#     final_df['safety_stock'].fillna(0) - final_df['Assumed Closing SOH_Lag_1'].fillna(0)

In [1006]:
# final_df['Calculated Primary Vol'] = final_df['Calculated Primary Vol'].clip(lower=0)

In [1007]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-06-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65095,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65096,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65097,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65098,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1008]:
calculated_primary = []

for (_, _), group_df in tqdm(final_df.groupby(['run_month', 'key'])):
    calculated_primary.extend(
        calculate_primary_iteratively(group_df)
    )

calculated_primary_df = pd.DataFrame(calculated_primary)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6510/6510 [00:14<00:00, 435.04it/s]


In [1009]:
del calculated_primary

In [1010]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    calculated_primary_df,
    on=['run_month', 'month_date', 'key'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [1011]:
final_df['Calculated Primary Vol'].min(), final_df['Final Assumed Closing SOH Vol'].min()

(0.0, 0.0)

In [1012]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [1013]:
final_df['Final Assumed Closing SOH Lag 1 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(1)

final_df['Final Assumed Closing SOH Lag 2 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(2)

In [1014]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3,Calculated Primary Vol,Final Assumed Closing SOH Vol,Final Assumed Closing SOH Lag 1 Vol,Final Assumed Closing SOH Lag 2 Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-05-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-06-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-07-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-08-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-06-30,2026-09-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65095,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
65096,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
65097,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
65098,Purplle_811416,Purplle,811416,SAF-MUSLI,2026-06-30,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [1015]:
brand_md_df

,brand_code,portfolio
0,ADV-AHO-R,Hair Oils
1,ADV-COL-R,Hair Oils
2,ADV-COL-S,Hair Oils
3,BD_BDOL_M,Male Grooming
4,BD_HRWX_M,Male Grooming
...,...,...
391,VEG_CLEAN,Health & Hygiene
392,VEG_CLN_G,Health & Hygiene
393,ZTK DEO,Male Grooming
394,ZTK GODEO,Youth


In [1016]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

In [1017]:
qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [1018]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [1019]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [1020]:
final_df.columns

Index(['key', 'chain', 'parent_material_code', 'material_group_code',
       'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'Offtake Chain PSKU', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol', 'Offtake P3M',
       'LY Offtake Actuals Vol', 'LY Offtake P3M',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       'LY Offtake Actuals Lag 3 Vol', '

In [1021]:
final_df = final_df.rename(columns={
    'key': 'Key',
    'chain': 'Chain',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'run_month': 'Run Month',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Till Date Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol',
    'Primary P3M': 'Primary P3M Vol',
    'Offtake Chain PSKU': 'Offtake Chain PSKU Vol',
    'norms_soh': 'Norms SOH',
    'norm_days': 'Norm Days',
    'safety_stock': 'Safety Stock Vol',
    'Actual Closing SOH': 'Actual Closing SOH Vol',
    'Actual Closing SOH_Lag_1': 'Actual Closing SOH Lag 1 Vol',
    'Actual Closing SOH Lag 2': 'Actual Closing SOH Lag 2 Vol',
    'Assumed Closing SOH': 'Assumed Closing SOH Vol',
    'Assumed Closing SOH_Lag_1': 'Assumed Closing SOH Lag 1 Vol',
    'Primary P3M redundant': 'Primary P3M redundant Vol',
    'LY Primary P3M': 'LY Primary P3M Vol',
    'Offtake P3M': 'Offtake P3M Vol',
    'LY Offtake P3M': 'LY Offtake P3M Vol',
    'Primary_Lag_1': 'Primary Actuals Lag 1 Vol',
    'Primary_Lag_2': 'Primary Actuals Lag 2 Vol',
    'Primary_Lag_3': 'Primary Actuals Lag 3 Vol',
    'OT_Lag_1': 'Offtake Actuals Lag 1 Vol',
    'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
    'OT_Lag_3': 'Offtake Actuals Lag 3 Vol',
    'qtr_ind_rate': 'Index Rate',
    'portfolio': 'Portfolio'
})

In [1022]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Offtake Chain PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2', 'Primary Actuals Vol', 'Sec Actuals Vol',
       'Primary P3M redundant Vol', 'LY Primary Actuals Vol',
       'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       

In [1023]:
final_df = final_df[[
    'Key', 'Chain', 'PSKU', 'Brand', 'Index Rate',
    'Portfolio', 'Run Month', 'Month Date',  'M month',

    'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
    'Primary P3M Vol', 'Offtake Chain PSKU Vol',

    'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
    'Safety Stock Vol', 
    
    'Actual Closing SOH Vol', 'Actual Closing SOH Lag 1 Vol', 
    'Actual Closing SOH Lag 2 Vol', 'Assumed Closing SOH Vol',
    'Assumed Closing SOH Lag 1 Vol', 'Final Assumed Closing SOH Vol', 
    'Final Assumed Closing SOH Lag 1 Vol', 
    
    'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
    'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
    'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
    'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
    'LY Primary Actuals Lead 2 Vol',

    'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol',
    'LY Offtake P3M Vol', 
    'Offtake Actuals Lag 1 Vol', 'Offtake Actuals Lag 2 Vol',
    'Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lag 1 Vol', 
    'LY Offtake Actuals Lag 2 Vol', 'LY Offtake Actuals Lag 3 Vol', 
    'LY Offtake Actuals Lead 1 Vol', 'LY Offtake Actuals Lead 2 Vol', 

    'Calculated Primary Vol'
]]

In [1024]:
vol_to_val_cols = [col for col in final_df.columns if 'Vol' in col]
vol_to_val_cols

['Primary Till Date Actuals Vol',
 'Secondary Plan Vol',
 'Primary P3M Vol',
 'Offtake Chain PSKU Vol',
 'Offtake Chain PSKU Forecast Vol',
 'Safety Stock Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Primary Actuals Vol',
 'Sec Actuals Vol',
 'Primary P3M redundant Vol',
 'LY Primary Actuals Vol',
 'LY Sec Actuals Vol',
 'LY Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'Offtake Actuals Vol',
 'Offtake P3M Vol',
 'LY Offtake Actuals Vol',
 'LY Offtake P3M Vol',
 'Offtake Actuals Lag 1 Vol',
 'Offtake Actuals Lag 2 Vol',
 'Offtake Actuals Lag 3 Vol',
 'LY Offtake Actuals L

In [1025]:
for col in vol_to_val_cols:
    final_df[col[:-3] + 'Val'] = final_df[col].fillna(0) * final_df['Index Rate'] / (10 ** 7)

In [1026]:
final_df

,Key,Chain,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,Primary Till Date Actuals Vol,...,LY Offtake P3M Val,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-06-30,2026-05-31,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-06-30,2026-06-30,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-06-30,2026-07-31,M+1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-06-30,2026-08-31,M+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-06-30,2026-09-30,M+3,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65095,Purplle_811416,Purplle,811416,SAF-MUSLI,315513.490535,Foods,2026-06-30,2026-10-31,M+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65096,Purplle_811416,Purplle,811416,SAF-MUSLI,315513.490535,Foods,2026-06-30,2026-11-30,M+5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65097,Purplle_811416,Purplle,811416,SAF-MUSLI,315513.490535,Foods,2026-06-30,2026-12-31,M+6,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65098,Purplle_811416,Purplle,811416,SAF-MUSLI,315513.490535,Foods,2026-06-30,2027-01-31,M+7,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1027]:
final_df.duplicated(subset=['Run Month', 'Key', 'Month Date']).sum()

0

In [1028]:
final_df['Calculated Primary Vol'].sum()

3564437.5892114453

### Depot PSKU

#### Aggregate to PSKU first

In [1029]:
psku_df = final_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [1030]:
psku_df = psku_df[psku_df['Month Date'] >= psku_df['Run Month']]

In [1031]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
1,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-06-30,0.0
2,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-07-31,0.0
3,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-08-31,0.0
4,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-09-30,0.0
5,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-10-31,0.0
...,...,...,...,...,...,...
9585,811416,SAF-MUSLI,Foods,2026-06-30,2026-10-31,0.0
9586,811416,SAF-MUSLI,Foods,2026-06-30,2026-11-30,0.0
9587,811416,SAF-MUSLI,Foods,2026-06-30,2026-12-31,0.0
9588,811416,SAF-MUSLI,Foods,2026-06-30,2027-01-31,0.0


In [1032]:
psku_df['Calculated Primary Vol'].sum()

3564437.589211446

In [1033]:
others_df = others_df.groupby(
    ['parent_material_code', 'material_group_code', 'month_date'], as_index=False
)['Primary P3M'].sum()

len_before_merge = len(others_df)
others_df = others_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on='material_group_code',
    how='left'
)
assert len_before_merge == len(others_df)
del len_before_merge

others_df = others_df.rename(columns={
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'Primary P3M': 'Calculated Primary Vol',
    'portfolio': 'Portfolio'
})

In [1034]:
others_df

,PSKU,Brand,Month Date,Calculated Primary Vol,Portfolio
0,715098,CO_SO_PCP,2023-07-31,0.0,Skin Care
1,715098,CO_SO_PCP,2023-08-31,0.0,Skin Care
2,715098,CO_SO_PCP,2023-09-30,0.0,Skin Care
3,715098,CO_SO_PCP,2023-10-31,0.0,Skin Care
4,715098,CO_SO_PCP,2023-11-30,0.0,Skin Care
...,...,...,...,...,...
28323,811416,SAF-MUSLI,2026-08-31,0.0,Foods
28324,811416,SAF-MUSLI,2026-09-30,0.0,Foods
28325,811416,SAF-MUSLI,2026-10-31,0.0,Foods
28326,811416,SAF-MUSLI,2026-11-30,0.0,Foods


In [1035]:
psku_df['Calculated Primary Vol'].min()

0.0

In [1036]:
final_others_forecast = pd.DataFrame()

for rm in psku_df['Run Month'].unique():
    tmp = others_df.copy()
    tmp['Run Month'] = rm
    tmp = tmp[tmp['Month Date'] <= psku_df[psku_df['Run Month'] == rm]['Month Date'].max()]
    tmp = tmp[tmp['Month Date'] >= rm]
    final_others_forecast = pd.concat([final_others_forecast, tmp], ignore_index=True)
    del tmp

In [1037]:
psku_df = pd.concat(
    [psku_df, final_others_forecast], ignore_index=True
)

In [1038]:
psku_df = psku_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [1039]:
psku_df['Calculated Primary Vol'].sum()

3580114.301544781

In [1040]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)

In [1041]:
depot_psku_primary_df

,DEPOT_CODE,PARENT_MATERIAL_CODE,MATERIAL_GROUP_CODE,MONTH_DATE,PRI_ACTUALS_VOL_RUM,PRI_APO_PLAN_VOL_RUM,SEC_APO_PLAN_VOL_RUM,SEC_ACTUALS_VOL_RUM
0,D111,718471,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
1,D111,718471,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
2,D111,718472,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
3,D111,718472,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
4,D111,718473,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
...,...,...,...,...,...,...,...,...
329988,D677,810179,SW_SGPRF,2026-04-30,0.0,13.416254,0.0,0.0
329989,D677,810179,SW_SGPRF,2026-05-31,0.0,19.109470,0.0,0.0
329990,D677,810179,SW_SGPRF,2026-06-30,0.0,21.831578,0.0,0.0
329991,D677,811169,SW_SGPRF,2026-03-31,0.0,9.818184,0.0,0.0


In [1042]:
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [1043]:
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')

In [1044]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [1045]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]

In [1046]:
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()

89

In [1047]:
depot_psku_primary_df.shape

(329204, 8)

In [1048]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
depot_psku_primary_df = depot_psku_primary_df.drop(indices_to_drop)

# Verify no duplicates remain
print(depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum())

0


In [1049]:
depot_psku_primary_df.shape

(329115, 8)

In [1050]:
depot_psku_primary_df = impute_missing_dates(
    depot_psku_primary_df.copy(),
    key=['depot_code', 'parent_material_code'],
    date_col='month_date'
)

19415it [00:04, 4339.68it/s]


In [1051]:
cols = ['depot_code', 'parent_material_code', 'material_group_code']

depot_psku_primary_df[cols] = depot_psku_primary_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [1052]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].fillna(0)

In [1053]:
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [1054]:
depot_psku_primary_df.duplicated(subset=['key', 'month_date']).sum()


0

In [1055]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)



pri_actuals_vol_rum
sec_actuals_vol_rum


In [1056]:
depot_psku_primary_df.sort_values(by=['key', 'month_date'], inplace=True)

In [1057]:
depot_psku_primary_df['Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [1058]:
depot_psku_primary_df['Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

depot_psku_primary_df['Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

depot_psku_primary_df['Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)

depot_psku_primary_df['LY Primary Actuals Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(12)

depot_psku_primary_df['LY Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(13)

depot_psku_primary_df['LY Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(14)

depot_psku_primary_df['LY Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(15)


depot_psku_primary_df['LY Primary Actuals Lead 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(11)

depot_psku_primary_df['LY Primary Actuals Lead 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(10)

In [1059]:
depot_psku_primary_df['LY Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['Primary P3M Vol'].shift(12)

In [1060]:
depot_psku_primary_df = depot_psku_primary_df.rename(columns={
    'key': 'Key',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Actuals Vol',
    'pri_apo_plan_vol_rum': 'Primary Plan Vol',
    'sec_actuals_vol_rum': 'Secondary Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol'
})

In [1061]:
depot_psku_primary_df

,Month Date,Key,Depot,PSKU,Brand,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,Secondary Actuals Vol,Primary P3M Vol,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,2017-04-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-05-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-06-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-07-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-08-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
980075,2026-08-31,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
980076,2026-09-30,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
980077,2026-10-31,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
980078,2026-11-30,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1062]:
final_depot_psku_df = depot_psku_primary_df[['Key', 'Depot', 'PSKU', 'Brand']].drop_duplicates()

In [1063]:
psku_df['Calculated Primary Vol'].sum()

3580114.301544781

In [1064]:
x = psku_df.copy()

In [1065]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = psku_df[psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['Brand'] != 'PABABY_ML'].index

# Remove those rows
psku_df = psku_df.drop(indices_to_drop)

# Verify no duplicates remain
print(psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date']).sum())

0


In [1066]:
psku_df['Calculated Primary Vol'].sum()

3579914.3015447804

In [1067]:
tmp_df = pd.DataFrame()

for rm in ['2026-06-30']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_depot_psku_df.copy()
        tmp_df2['Run Month'] = pd.to_datetime(rm)
        tmp_df2['Month Date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_depot_psku_df = tmp_df.copy()    
del tmp_df

In [1068]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
0,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-06-30,0.0
1,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-07-31,0.0
2,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-08-31,0.0
3,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-09-30,0.0
4,715098,CO_SO_PCP,Skin Care,2026-06-30,2026-10-31,0.0
...,...,...,...,...,...,...
8668,811416,SAF-MUSLI,Foods,2026-06-30,2026-10-31,0.0
8669,811416,SAF-MUSLI,Foods,2026-06-30,2026-11-30,0.0
8670,811416,SAF-MUSLI,Foods,2026-06-30,2026-12-31,0.0
8671,811416,SAF-MUSLI,Foods,2026-06-30,2027-01-31,0.0


In [1069]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    psku_df[['PSKU', 'Run Month', 'Month Date', 'Calculated Primary Vol']], 
    on=['PSKU', 'Run Month', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1070]:
depot_psku_primary_df.columns

Index(['Month Date', 'Key', 'Depot', 'PSKU', 'Brand', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol'],
      dtype='object')

In [1071]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    depot_psku_primary_df[['Depot', 'PSKU', 'Month Date', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',  'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 
       'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol']], 
    on=['Depot', 'PSKU', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1072]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194605,D677_811268,D677,811268,PA_ESS_HO,2026-06-30,2027-02-28,0.184,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194606,D677_811269,D677,811269,PA_ESS_HO,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194607,D677_811279,D677,811279,SAF_CDPRS,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194608,D677_811287,D677,811287,PA_RSW_SR,2026-06-30,2027-02-28,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1073]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol'],
      dtype='object')

In [1074]:
final_depot_psku_df['Primary P3M copy Vol'] = final_depot_psku_df['Primary P3M Vol'].copy()

In [1075]:
final_depot_psku_df['Primary P3M Vol'] = final_depot_psku_df['Primary P3M Vol'].fillna(0)

In [1076]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194605,D677_811268,D677,811268,PA_ESS_HO,2026-06-30,2027-02-28,0.184,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194606,D677_811269,D677,811269,PA_ESS_HO,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194607,D677_811279,D677,811279,SAF_CDPRS,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194608,D677_811287,D677,811287,PA_RSW_SR,2026-06-30,2027-02-28,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1077]:
final_depot_psku_df['PSKU Primary P3M Sum Vol'] = final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['Primary P3M Vol'].transform('sum')

In [1078]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194605,D677_811268,D677,811268,PA_ESS_HO,2026-06-30,2027-02-28,0.184,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
194606,D677_811269,D677,811269,PA_ESS_HO,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
194607,D677_811279,D677,811279,SAF_CDPRS,2026-06-30,2027-02-28,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
194608,D677_811287,D677,811287,PA_RSW_SR,2026-06-30,2027-02-28,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1079]:
final_depot_psku_df.sort_values(by=['Run Month', 'Key', 'Month Date'], inplace=True)

In [1080]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [1081]:
for col in ['Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 
            'Primary Actuals Lag 3 Vol', 'PSKU Primary P3M Sum Vol']:
    # if not 'LY' in col:  'LY P6M',
    final_depot_psku_df.loc[final_depot_psku_df['Month Date'] > final_depot_psku_df['Run Month'], [col]] = np.nan
    final_depot_psku_df[col] = final_depot_psku_df.groupby(['Run Month', 'Key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [1082]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19461,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38922,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58383,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
77844,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116765,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-10-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
136226,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
155687,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
175148,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1083]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [1084]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19461,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38922,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58383,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
77844,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116765,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-10-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
136226,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
155687,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
175148,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1085]:
final_depot_psku_df['PSKU P3M Contribution'] = final_depot_psku_df['Primary P3M Vol'] / final_depot_psku_df['PSKU Primary P3M Sum Vol']

In [1086]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'] = final_depot_psku_df['Calculated Primary Vol'] \
    * final_depot_psku_df['PSKU P3M Contribution']

In [1087]:
final_depot_psku_df.rename(columns={'Calculated Primary Vol': 'Calculated PSKU Primary Vol'}, inplace=True)

In [1088]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'].sum()

3399387.7722503496

In [1089]:
vol_to_val_cols = [col for col in final_depot_psku_df.columns if ('Vol' in col) and ('copy' not in col)]
vol_to_val_cols

['Calculated PSKU Primary Vol',
 'Primary Actuals Vol',
 'Primary Plan Vol',
 'Secondary Plan Vol',
 'Secondary Actuals Vol',
 'Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'LY Primary P3M Vol',
 'PSKU Primary P3M Sum Vol',
 'Calculated Depot PSKU Primary Vol']

In [1090]:
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [1091]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1092]:
for col in vol_to_val_cols:
    final_depot_psku_df[col[:-3] + 'Val'] = final_depot_psku_df[col] * final_depot_psku_df['Index Rate'] / (10 ** 7)

In [1093]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Val,LY Primary Actuals Val,LY Primary Actuals Lag 1 Val,LY Primary Actuals Lag 2 Val,LY Primary Actuals Lag 3 Val,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val
0,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-05-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,D111_709567,D111,709567,SAFF OATS,2026-06-30,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194605,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-10-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
194606,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
194607,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
194608,D677_811416,D677,811416,SAF-MUSLI,2026-06-30,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN


In [1094]:
final_depot_psku_df['Calculated Depot PSKU Primary Val'] = final_depot_psku_df['Calculated Depot PSKU Primary Val'].fillna(0)

In [1095]:
final_depot_psku_df['M Month'] = final_depot_psku_df.apply(
    lambda x: mappings[x['Run Month']].get(x['Month Date'], np.nan),
    axis=1
)

In [1096]:
final_depot_psku_df = final_depot_psku_df[final_depot_psku_df['M Month'].notna()]

In [1097]:
final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['PSKU P3M Contribution'].max().max()

1.0

In [1098]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 'Run Month',
       'Month Date', 'M month', 'Primary Till Date Actuals Vol',
       'Secondary Plan Vol', 'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'Off

In [1099]:
final_df['Calculated Primary Val'].sum(), final_depot_psku_df['Calculated Depot PSKU Primary Val'].sum()

(360.81544615079014, 353.3366255646542)

In [1100]:
final_df[final_df['Month Date']=='2026-07-31']['Calculated Primary Val'].sum()

39.940347982274375

In [1101]:
final_depot_psku_df[final_depot_psku_df['Month Date']=='2026-07-31']['Calculated Depot PSKU Primary Val'].sum()

39.67468626389758

## SAVE

In [1102]:
VERSION = 'Jun26 Live Run' 

In [1103]:
os.makedirs(f'FINAL ECOM Chain PSKU OTP/{VERSION}')

In [1104]:
final_depot_psku_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Depot PSKU Primary_{VERSION}.csv', index=False)
final_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Chain PSKU Primary_{VERSION}.csv', index=False)

In [311]:
norm_days.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norm Days {VERSION}.csv', index=False)
norms.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norms {VERSION}.csv', index=False)

In [312]:
soh_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/SOH {VERSION}.csv', index=False)